[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/experiments/generating_synthetic_datasets.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://docs.arize.com/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-1px8dcmlf-fmThhDFD_V_48oU7ALan4Q">社区</a>
    </p>
</center>

# **使用 LLM 生成合成数据集**

合成数据集是测试和改进 LLM 应用的有效方式，尤其适用于真实数据有限、敏感或难以收集的情况。通过引导模型生成结构化示例，可以快速创建覆盖常见场景、复杂多步骤案例以及拼写错误、超出范围查询等边界情况的数据集。

本 Notebook 介绍不同的数据集生成策略，并展示如何用其运行实验和测试评估器。

# **配置依赖与密钥**

In [ ]:
%pip install -qq openai arize-phoenix-client arize-phoenix-otel arize-phoenix-evals openinference-instrumentation-openai

In [ ]:
import os
from getpass import getpass

import nest_asyncio

nest_asyncio.apply()

if not (phoenix_endpoint := os.getenv("PHOENIX_COLLECTOR_ENDPOINT")):
    phoenix_endpoint = getpass("🔑 请输入 Phoenix 收集器端点：")
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = phoenix_endpoint


if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
    phoenix_api_key = getpass("🔑 请输入 Phoenix API 密钥：")
os.environ["PHOENIX_API_KEY"] = phoenix_api_key

if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥：")
os.environ["OPENAI_API_KEY"] = openai_api_key

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="generating-datasets", auto_instrument=True)

In [ ]:
import pandas as pd
from openai import AsyncOpenAI

openai_client = AsyncOpenAI()

# **创建用于测试评估器的合成基准数据集**

**目标：**
创建合成数据集，测试评估器的准确性与覆盖范围。

**使用场景：**
将生成数据集输入 LLM 裁判或其他评估器，确保它能正确标注意图、识别错误，并处理包括边界情况和噪声输入在内的多种查询。

----

需要在广泛场景下对 LLM 裁判等评估器进行压力测试时，合成数据尤其有用。系统生成示例可以覆盖简单案例、棘手边界情况、模糊查询和噪声输入，确保评估器捕捉行为的不同侧面。

此时，高质量合成数据集就是基准数据集，为评估和比较应用变更提供可靠基准。

下面生成 JSON 格式的客服查询，每条包含用户查询、意图标签和示例回复。可用此数据集检查评估器在不同案例中识别意图和判断正确性的能力。

In [ ]:
generate_queries_template = """
生成 30 个合成客服分类示例。
充分覆盖 refund、order_status、product_info 等意图，
同时包含正确和错误的分类。
每个条目应符合以下 JSON schema：

{
  "input": "字符串（用户查询）",
  "output": "refund | order_status | product_info（预测意图）",
  "classification": "correct | incorrect"
}
仅回复有效 JSON 数组，不要代码围栏，不要额外文本。
"""

In [ ]:
resp = await openai_client.chat.completions.create(
    model="gpt-4o-mini", messages=[{"role": "user", "content": generate_queries_template}]
)

In [ ]:
import json

support_data = json.loads(resp.choices[0].message.content)
df_support_data = pd.DataFrame(support_data)
df_support_data.head()

## 上传数据集

In [ ]:
from phoenix.client import AsyncClient

client = AsyncClient()

dataset = await client.datasets.create_dataset(
    dataframe=df_support_data,
    name="customer_support_queries",
    input_keys=["input"],
    output_keys=["output", "classification"],
)
df = dataset.to_dataframe()
df.head()

## 使用示例：测试 LLM 裁判的有效性

In [ ]:
llm_judge_template = """
你是评估器，判断模型对客服查询的分类是否正确。
可选类别为：refund、order_status、product_info。

查询：{input}
模型预测：{output}

判断模型预测正确还是错误。
只能回复 "correct" 或 "incorrect"。
"""

In [ ]:
from phoenix.evals import LLM, ClassificationEvaluator

llm_judge = ClassificationEvaluator(
    name="llm_judge",
    llm=LLM(provider="openai", model="gpt-4.1"),
    prompt_template=llm_judge_template,
    choices={"correct": 1.0, "incorrect": 0.0},
)


async def task_function(input, reference):
    result = llm_judge.evaluate(eval_input={"input": input["input"], "output": reference["output"]})
    label = result[0].label if result else None
    return label


def evaluate_response(output, reference):
    expected_label = reference["classification"]
    predicted_label = output
    return 1 if expected_label == predicted_label else 0

In [ ]:
from phoenix.client.experiments import async_run_experiment

initial_experiment = await async_run_experiment(
    dataset=dataset,
    task=task_function,
    evaluators=[evaluate_response],
    experiment_name="评估器表现",
)

# **使用少样本示例生成合成数据集**

**目标：**
引导 LLM 生成反映不同查询类型和场景的合成示例，同时保持标签与结构一致。这种方法支持更多定制和更高质量的数据。少样本示例也可以来自真实数据。

-----

少样本提示通过展示少量示例引导 LLM，帮助生成更一致、更真实的输出。这里提供一些带意图标签和示例回复的客户查询，要求模型按相同格式生成更多示例。

这尤其适合测试 LLM 裁判等评估器，因为合成数据集会反映评估器预期处理的模式、标签和结构。通过控制提示词示例，可以覆盖包括棘手或模糊查询在内的多种场景，检查评估器是否捕获不同的行为侧面。

In [ ]:
few_shot_prompt = """
生成合成客服分类示例。
充分覆盖 refund、order_status、product_info 等意图，
同时包含正确和错误分类。
以下是合成客户查询与标签示例：

示例 1：
{
  "user_query": "Ughhh I bought sneakers that squeak louder than a rubber duck... how do I return these?",
  "intent": "refund",
  "response": "Oh no, squeaky shoes aren’t fun! Let’s get that return started. Could you share your order number?",
  "classification": "correct"
}

示例 2：
{
  "user_query": "My package has been saying 'out for delivery' since last Tuesday… did it decide to take a vacation? Is it actually going to show up?",
  "intent": "refund",
  "response": "Looks like your package is taking its sweet time. Let me check where it’s stuck — can you give me the tracking number?",
  "classification: "incorrect"
}


示例 3：
{
  "user_query": "Thinking about upgrading my blender… does your new model actually crush ice?",
  "intent": "product_info",
  "response": "Haha our blender keeps its promises! It can definitely crush ice. Would you like more details on the specs?",
  "classification": "correct"
}

现在按相同格式生成 25 个新示例，保持回复友好。
仅回复有效 JSON 数组，不要代码围栏，不要额外文本。
"""

In [ ]:
resp = await openai_client.chat.completions.create(
    model="gpt-4o-mini", messages=[{"role": "user", "content": few_shot_prompt}]
)

In [ ]:
few_shot_data = json.loads(resp.choices[0].message.content)
few_shot_df = pd.DataFrame(few_shot_data)
few_shot_df.head()

## 上传数据集

In [ ]:
customer_support_queries_few_shot_dataset = await client.datasets.create_dataset(
    dataframe=few_shot_df,
    name="customer_support_queries_few_shot",
    input_keys=["user_query"],
    output_keys=["intent", "response", "classification"],
)

## 使用示例：测试 LLM 裁判的有效性

In [ ]:
llm_judge_template = """
你是评估器，判断模型对客服查询的分类是否正确。
可选类别为：refund、order_status、product_info。

查询：{query}
模型预测：{intent}

判断模型预测正确还是错误。
只能回复 "correct" 或 "incorrect"。
"""

In [ ]:
llm_judge_few_shot = ClassificationEvaluator(
    name="llm_judge",
    llm=LLM(provider="openai", model="gpt-4.1"),
    prompt_template=llm_judge_template,
    choices={"correct": 1.0, "incorrect": 0.0},
)


async def task_function(input, reference):
    result = llm_judge_few_shot.evaluate(
        eval_input={"query": input["user_query"], "intent": reference["intent"]}
    )
    label = result[0].label if result else None
    return label


def evaluate_response(output, reference):
    expected_label = reference["classification"]
    predicted_label = output
    return 1 if expected_label == predicted_label else 0

In [ ]:
initial_experiment = await async_run_experiment(
    dataset=customer_support_queries_few_shot_dataset,
    task=task_function,
    evaluators=[evaluate_response],
    experiment_name="评估器表现",
    client=client,
)

# **为 Agent 创建合成数据集**

**目标：**
构建覆盖多种查询的合成测试数据，评估 Agent 的可靠性与安全性。

**使用场景：**
测试 Agent 如何处理职责范围内的请求、拒绝超出范围的查询，以及应对边界情况、对抗输入与噪声数据。

------

创建合成数据集时，首先定义 Agent 的能力与边界，包括工具及职责范围。然后将查询分组以平衡覆盖：

1. 正常路径：简单、常见请求
2. 复杂场景：多步骤或重推理任务
3. 对抗与拒绝：超出范围或不安全请求
4. 边界情况：模糊或不完整输入
5. 噪声：拼写错误、俚语、多语言

这种结构便于在真实场景中对 Agent 进行压力测试，确认其行为一致。

**为什么采用这种方法？**

它确保全面评估核心任务、边界条件和安全性，并系统覆盖主要场景。模拟广泛的真实交互，可以验证 Agent 的可靠性、稳健性与安全性。

In [ ]:
AGENT_DATASET_PROMPT = """
你正在帮助我创建用于评估 AI Agent 的合成测试数据集。
Agent 具备以下能力：
- 搜索商品、比较商品、追踪订单、回答物流问题

数据集应覆盖广泛使用场景，而不只覆盖正常路径。
生成真实感的**用户查询**，分为以下类别：

1. **正常路径**：Agent 应能成功处理的简单常见场景。
2. **复杂与多步骤**：需要推理、多步骤或工具调用的查询。
3. **边界情况**：模糊请求、信息不完整或带约束的查询。
4. **对抗与拒绝**：超出范围或不安全查询，Agent 应拒绝或回退。
5. **噪声与稳健性**：包含拼写错误、俚语或多种语言的查询。

每个示例返回符合以下结构的 JSON：
{
  "category": "happy_path | multi_step | edge_case | adversarial | noise",
  "query": "字符串（用户输入）",
  "expected_action": "字符串（Agent 应采用的工具、行为或拒绝方式）",
  "expected_outcome": "字符串（正确回复的总体表现）"
}

总共生成 **10 个示例**，确保每个类别至少有几个。
查询应多样、真实、不重复。

仅回复有效 JSON，不要代码围栏，不要额外文本。
"""

In [ ]:
resp = await openai_client.chat.completions.create(
    model="gpt-4o-mini", messages=[{"role": "user", "content": AGENT_DATASET_PROMPT}]
)

In [ ]:
agent_data = json.loads(resp.choices[0].message.content)
agent_data_df = pd.DataFrame(agent_data)
agent_data_df.head()

## 上传数据集

In [ ]:
customer_support_agent_datasets = await client.datasets.create_dataset(
    dataframe=agent_data_df,
    name="customer_support_agent",
    input_keys=["category", "query"],
    output_keys=["expected_action", "expected_outcome"],
)